# Fundamentos probabilísticos para filtros de Kalman
## Densidades, gaussianas, momentos, probabilidad condicional, Bayes y límite central

Este cuaderno desarrolla, con ejemplos reproducibles en Python, seis ideas presentadas en **KalmanFilters_1.pdf**. No deriva todavía el filtro de Kalman; construye el lenguaje probabilístico necesario para entenderlo.

### Objetivos

1. Interpretar y normalizar una función de densidad de probabilidad.
2. Trabajar con distribuciones gaussianas univariables y bivariables.
3. Calcular valor esperado, varianza y matriz de covarianza.
4. Comprender una probabilidad y una densidad condicional.
5. Aplicar el teorema de Bayes a eventos y a una medición gaussiana.
6. Observar experimentalmente el teorema del límite central.

> **Convención:** las mayúsculas, como $X$, denotan variables aleatorias; las minúsculas, como $x$, denotan valores posibles o realizaciones.


## Mapa conceptual

| Concepto | Pregunta que responde | Papel posterior en Kalman |
|---|---|---|
| Densidad $f_X(x)$ | ¿Dónde son más plausibles los valores de $X$? | Modela incertidumbre continua |
| Gaussiana | ¿Cómo describimos incertidumbre mediante media y dispersión? | Modelo habitual del estado y del ruido |
| Esperanza y covarianza | ¿Cuál es el centro y cómo varían juntas las componentes? | Vectores de estado y matrices $P$, $Q$ y $R$ |
| Condicionamiento | ¿Qué sabemos de $X$ cuando observamos $Y=y$? | Estimación dada una medición |
| Bayes | ¿Cómo combina una observación la información previa y la verosimilitud? | Fundamento de la actualización de medición |
| Límite central | ¿Por qué aparecen gaussianas al sumar muchos efectos? | Motivación parcial del modelo de ruido gaussiano |


## Preparación del entorno

El tutorial utiliza únicamente NumPy y Plotly. Si hace falta instalarlos, ejecuta una vez:

```bash
pip install numpy plotly
```


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(seed=2026)


def normal_pdf(x, mu=0.0, sigma=1.0):
    """Densidad de N(mu, sigma^2), evaluada elemento a elemento."""
    if sigma <= 0:
        raise ValueError('sigma debe ser positiva.')
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (np.sqrt(2 * np.pi) * sigma)


# Compatible con versiones antiguas y recientes de NumPy.
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


AttributeError: module 'numpy' has no attribute 'trapz'

# 1. Función de densidad de probabilidad

Para una variable aleatoria continua $X$, la función de densidad $f_X(x)$ debe satisfacer

$$f_X(x)\ge 0,\qquad \int_{-\infty}^{\infty} f_X(x)\,dx=1.
$$

La probabilidad de que $X$ caiga en un intervalo es el área bajo la densidad:

$$
P(a\le X\le b)=\int_a^b f_X(x)\,dx.
$$

Un punto importante es que **densidad no es lo mismo que probabilidad**. Para una variable continua, $P(X=x_0)=0$, aunque $f_X(x_0)$ pueda ser grande. La densidad tiene unidades inversas a las de $X$; la integral es adimensional y sí representa una probabilidad.

### Ejemplo: área bajo una densidad normal estándar

Aproximaremos numéricamente la integral total y $P(-1\le X\le1)$.


In [ ]:
x = np.linspace(-5.0, 5.0, 5_001)
fx = normal_pdf(x)
mask = (x >= -1.0) & (x <= 1.0)

total_area = integrate(fx, x)
prob_minus1_plus1 = integrate(fx[mask], x[mask])

print(f'Integral aproximada en [-5, 5]: {total_area:.8f}')
print(f'P(-1 ≤ X ≤ 1):                {prob_minus1_plus1:.4%}')

assert np.isclose(total_area, 1.0, atol=1e-5)
assert np.isclose(prob_minus1_plus1, 0.6827, atol=2e-4)

fig_pdf = go.Figure()
fig_pdf.add_trace(go.Scatter(x=x, y=fx, mode='lines', name='f_X(x)', line=dict(width=3)))
fig_pdf.add_trace(go.Scatter(
    x=x[mask], y=fx[mask], mode='lines', fill='tozeroy',
    name='Área: P(-1 ≤ X ≤ 1)', line=dict(color='darkorange')
))
fig_pdf.update_layout(
    title='La probabilidad de un intervalo es un área bajo la densidad',
    xaxis_title='x', yaxis_title='Densidad', template='plotly_white',
    width=850, height=480
)
fig_pdf.show()


# 2. Distribución gaussiana univariable y de dos variables

## 2.1 Gaussiana univariable

Se escribe $X\sim\mathcal N(\mu,\sigma^2)$ y su densidad es

$$
f_X(x)=\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left[-\frac{(x-\mu)^2}{2\sigma^2}\right].
$$

- $\mu$ desplaza el centro de la campana.
- $\sigma^2$ es la varianza y controla la dispersión.
- La distribución es simétrica alrededor de $\mu$.
- Aproximadamente 68.27 %, 95.45 % y 99.73 % cae dentro de 1, 2 y 3 desviaciones estándar de la media, respectivamente.

El panel izquierdo compara parámetros. El derecho contrasta un histograma de muestras con la densidad que las generó.


In [ ]:
x_uni = np.linspace(-6.0, 8.0, 1_200)
samples_uni = rng.normal(loc=2.0, scale=1.5, size=20_000)

fig_uni = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Efecto de μ y σ', 'Muestras frente a densidad teórica')
)

for mu_i, sigma_i, label in [
    (0.0, 1.0, 'N(0, 1)'),
    (2.0, 1.0, 'N(2, 1)'),
    (0.0, 2.0, 'N(0, 4)'),
]:
    fig_uni.add_trace(
        go.Scatter(x=x_uni, y=normal_pdf(x_uni, mu_i, sigma_i), mode='lines', name=label),
        row=1, col=1
    )

fig_uni.add_trace(
    go.Histogram(x=samples_uni, histnorm='probability density', nbinsx=80,
                 opacity=0.55, name='Muestras N(2, 1.5²)'),
    row=1, col=2
)
fig_uni.add_trace(
    go.Scatter(x=x_uni, y=normal_pdf(x_uni, 2.0, 1.5), mode='lines',
               line=dict(color='black', width=3), name='Densidad teórica'),
    row=1, col=2
)
fig_uni.update_layout(
    title='Distribución gaussiana univariable', template='plotly_white',
    width=1_050, height=480, barmode='overlay'
)
fig_uni.update_xaxes(title_text='x')
fig_uni.update_yaxes(title_text='Densidad')
fig_uni.show()


## 2.2 Gaussiana de dos variables

Para $X=[X_1,X_2]^\top$, escribimos

$$X\sim\mathcal N(\mu,\Sigma),$$

con densidad

$$
f_X(x)=\frac{1}{2\pi|\Sigma|^{1/2}}
\exp\!\left[-\frac{1}{2}(x-\mu)^\top
\Sigma^{-1}(x-\mu)\right].
$$

La media $\mu$ fija el centro. La covarianza $\Sigma$ fija la dispersión, orientación y correlación. Las curvas de densidad constante son elipses determinadas por la distancia de Mahalanobis

$$d_M^2=(x-\mu)^\top\Sigma^{-1}(x-\mu).$$

Usaremos $\mu=[1,2]^\top$ y $\Sigma=\begin{bmatrix}1&0.7\\0.7&1.5\end{bmatrix}$.


In [ ]:
mu_bi = np.array([1.0, 2.0])
Sigma_bi = np.array([[1.0, 0.7],
                     [0.7, 1.5]])

def bivariate_normal_pdf(x1, x2, mu, Sigma):
    """Densidad normal bivariada sobre arreglos x1 y x2."""
    pos = np.stack((x1 - mu[0], x2 - mu[1]), axis=-1)
    inv_Sigma = np.linalg.inv(Sigma)
    exponent = np.einsum('...i,ij,...j->...', pos, inv_Sigma, pos)
    normalizer = 2.0 * np.pi * np.sqrt(np.linalg.det(Sigma))
    return np.exp(-0.5 * exponent) / normalizer

grid_1 = np.linspace(-3.0, 5.0, 170)
grid_2 = np.linspace(-3.0, 7.0, 170)
X1, X2 = np.meshgrid(grid_1, grid_2)
F12 = bivariate_normal_pdf(X1, X2, mu_bi, Sigma_bi)

fig_bi = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'surface'}, {'type': 'xy'}]],
    subplot_titles=('Superficie de densidad', 'Curvas de nivel')
)
fig_bi.add_trace(
    go.Surface(x=grid_1, y=grid_2, z=F12, colorscale='Viridis', showscale=False),
    row=1, col=1
)
fig_bi.add_trace(
    go.Contour(x=grid_1, y=grid_2, z=F12, colorscale='Viridis',
               contours=dict(coloring='lines'), showscale=False),
    row=1, col=2
)
fig_bi.add_trace(
    go.Scatter(x=[mu_bi[0]], y=[mu_bi[1]], mode='markers',
               marker=dict(size=11, color='red'), name='Media'),
    row=1, col=2
)
fig_bi.update_layout(
    title='Densidad normal bivariada', template='plotly_white',
    width=1_050, height=550
)
fig_bi.update_xaxes(title_text='x₁', row=1, col=2)
fig_bi.update_yaxes(title_text='x₂', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig_bi.show()


# 3. Valor esperado y covarianza

## 3.1 Valor esperado

Para una variable continua,

$$\mathbb E[X]=\int_{-\infty}^{\infty}x f_X(x)\,dx.
$$

Es el centro de masa probabilístico, no necesariamente un valor que la variable deba tomar. Para muestras $x^{(1)},\ldots,x^{(N)}$, se aproxima mediante

$$\hat\mu=\frac{1}{N}\sum_{k=1}^{N}x^{(k)}.
$$

## 3.2 Varianza y covarianza

La varianza mide dispersión alrededor de la media:

$$\operatorname{Var}(X)=\mathbb E[(X-\mu)^2].$$

Para un vector, la matriz de covarianza es

$$\Sigma=\mathbb E[(X-\mu)(X-\mu)^\top].$$

Sus elementos diagonales son varianzas y los elementos fuera de la diagonal describen variación lineal conjunta. Debe ser simétrica y semidefinida positiva.


In [ ]:
n_samples = 10_000
L = np.linalg.cholesky(Sigma_bi)
Z = rng.standard_normal((2, n_samples))
samples_bi = mu_bi[:, None] + L @ Z

sample_mean = np.mean(samples_bi, axis=1)
sample_cov = np.cov(samples_bi)
sample_corr = np.corrcoef(samples_bi)[0, 1]
theoretical_corr = Sigma_bi[0, 1] / np.sqrt(Sigma_bi[0, 0] * Sigma_bi[1, 1])

print('Media teórica:              ', mu_bi)
print('Media muestral:             ', sample_mean)
print('\nCovarianza teórica:\n', Sigma_bi)
print('\nCovarianza muestral:\n', sample_cov)
print(f'\nCorrelación teórica:  {theoretical_corr:.4f}')
print(f'Correlación muestral: {sample_corr:.4f}')

assert np.allclose(sample_mean, mu_bi, atol=0.05)
assert np.allclose(sample_cov, Sigma_bi, atol=0.06)

fig_samples = go.Figure(go.Scattergl(
    x=samples_bi[0], y=samples_bi[1], mode='markers',
    marker=dict(size=4, opacity=0.25, color='royalblue'),
    name='Muestras'
))
fig_samples.add_trace(go.Scatter(
    x=[sample_mean[0]], y=[sample_mean[1]], mode='markers',
    marker=dict(size=12, color='gold', line=dict(color='black', width=1)),
    name='Media muestral'
))
fig_samples.update_layout(
    title='Muestras de la gaussiana bivariada',
    xaxis_title='X₁', yaxis_title='X₂', template='plotly_white',
    width=750, height=600, yaxis=dict(scaleanchor='x', scaleratio=1)
)
fig_samples.show()


# 4. Probabilidad condicional

Para eventos $A$ y $B$ con $P(B)>0$,

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}.
$$

Condicionar significa restringir el universo a los casos donde $B$ ocurrió y volver a normalizar.

### Ejemplo discreto con dos dados

Sean $A=$ «la suma es al menos 9» y $B=$ «el primer dado es 4, 5 o 6». Enumeraremos los 36 resultados equiprobables.


In [ ]:
outcomes = [(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)]
A = [(d1, d2) for d1, d2 in outcomes if d1 + d2 >= 9]
B = [(d1, d2) for d1, d2 in outcomes if d1 >= 4]
A_and_B = [outcome for outcome in outcomes if outcome in A and outcome in B]

P_A = len(A) / len(outcomes)
P_B = len(B) / len(outcomes)
P_A_and_B = len(A_and_B) / len(outcomes)
P_A_given_B = P_A_and_B / P_B

print(f'P(A)       = {len(A)}/36 = {P_A:.4f}')
print(f'P(B)       = {len(B)}/36 = {P_B:.4f}')
print(f'P(A ∩ B)   = {len(A_and_B)}/36 = {P_A_and_B:.4f}')
print(f'P(A | B)   = {len(A_and_B)}/{len(B)} = {P_A_given_B:.4f}')

assert np.isclose(P_A_given_B, len(A_and_B) / len(B))


## 4.1 Condicionamiento de una gaussiana bivariada

La versión continua es

$$f_{X_2\mid X_1}(x_2\mid x_1)=\frac{f_{X_1,X_2}(x_1,x_2)}{f_{X_1}(x_1)}.
$$

Si el vector es conjuntamente gaussiano, la distribución condicional también es gaussiana. Separando

$$
\mu=\begin{bmatrix}\mu_1\\\mu_2\end{bmatrix},\qquad
\Sigma=\begin{bmatrix}\Sigma_{11}&\Sigma_{12}\\\Sigma_{21}&\Sigma_{22}\end{bmatrix},
$$

se obtiene

$$
X_2\mid X_1=x_1\sim\mathcal N\!\left(
\mu_2+\Sigma_{21}\Sigma_{11}^{-1}(x_1-\mu_1),
\Sigma_{22}-\Sigma_{21}\Sigma_{11}^{-1}\Sigma_{12}
\right).
$$

Observar $X_1$ desplaza la media de $X_2$ y reduce su incertidumbre cuando existe correlación.


In [ ]:
x1_observed = 2.0
mu1, mu2 = mu_bi
S11 = Sigma_bi[0, 0]
S12 = Sigma_bi[0, 1]
S21 = Sigma_bi[1, 0]
S22 = Sigma_bi[1, 1]

conditional_mean = mu2 + (S21 / S11) * (x1_observed - mu1)
conditional_var = S22 - S21 * S12 / S11
conditional_std = np.sqrt(conditional_var)

y_grid = np.linspace(-2.0, 7.0, 1_000)
marginal_y2 = normal_pdf(y_grid, mu2, np.sqrt(S22))
conditional_y2 = normal_pdf(y_grid, conditional_mean, conditional_std)

print(f'Antes de observar X₁: E[X₂]={mu2:.3f}, Var(X₂)={S22:.3f}')
print(f'Dado X₁={x1_observed:.1f}: E[X₂|X₁]={conditional_mean:.3f}, '
      f'Var(X₂|X₁)={conditional_var:.3f}')

assert conditional_var < S22

fig_cond = go.Figure()
fig_cond.add_trace(go.Scatter(
    x=y_grid, y=marginal_y2, mode='lines', line=dict(width=3),
    name='Marginal: X₂'
))
fig_cond.add_trace(go.Scatter(
    x=y_grid, y=conditional_y2, mode='lines', line=dict(width=3),
    name=f'Condicional: X₂ | X₁={x1_observed}'
))
fig_cond.update_layout(
    title='Condicionar desplaza la media y reduce la varianza',
    xaxis_title='x₂', yaxis_title='Densidad', template='plotly_white',
    width=850, height=480
)
fig_cond.show()


# 5. Teorema de Bayes

A partir de $P(A\cap B)=P(B\mid A)P(A)=P(A\mid B)P(B)$, obtenemos

$$P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}.
$$

En términos de una cantidad desconocida $x$ y una observación $z$,

$$
p(x\mid z)=\frac{p(z\mid x)p(x)}{p(z)}.
$$

- $p(x)$: distribución **previa** o *prior*.
- $p(z\mid x)$: **verosimilitud** de la observación.
- $p(z)$: **evidencia**, que normaliza el resultado.
- $p(x\mid z)$: distribución **posterior**.

## 5.1 Ejemplo discreto: detector con falsas alarmas

Supongamos que un blanco está presente en 20 % de los casos, el detector acierta 90 % cuando hay blanco y produce una falsa alarma 10 % cuando no lo hay. ¿Qué probabilidad hay de que realmente exista un blanco después de una detección?


In [ ]:
P_T = 0.20                 # P(blanco presente)
P_not_T = 1.0 - P_T
P_D_given_T = 0.90         # Probabilidad de detección
P_D_given_not_T = 0.10     # Probabilidad de falsa alarma

P_D = P_D_given_T * P_T + P_D_given_not_T * P_not_T
P_T_given_D = P_D_given_T * P_T / P_D

print(f'P(detección) = {P_D:.3f}')
print(f'P(blanco | detección) = {P_T_given_D:.3%}')
print('Aunque el detector acierta 90 %, una detección no implica una certeza del 90 %.')

assert np.isclose(P_T_given_D, 0.18 / 0.26)

fig_bayes_discrete = go.Figure(go.Bar(
    x=['Prior: P(blanco)', 'Posterior: P(blanco | detección)'],
    y=[P_T, P_T_given_D],
    text=[f'{P_T:.1%}', f'{P_T_given_D:.1%}'],
    textposition='outside', marker_color=['slateblue', 'darkorange']
))
fig_bayes_discrete.update_layout(
    title='La detección actualiza la probabilidad previa',
    yaxis_title='Probabilidad', yaxis_range=[0, 1],
    template='plotly_white', width=750, height=480
)
fig_bayes_discrete.show()


## 5.2 Ejemplo continuo: actualización gaussiana

Sea un estado escalar con prior $X\sim\mathcal N(\mu_0,P_0)$ y una medición

$$Z=X+V,\qquad V\sim\mathcal N(0,R).$$

Para una medición observada $z$, la posterior es gaussiana:

$$
P^+=\left(P_0^{-1}+R^{-1}\right)^{-1},\qquad
\mu^+=P^+\left(P_0^{-1}\mu_0+R^{-1}z\right).
$$

Esta es la esencia de una actualización escalar de Kalman: combinar dos fuentes de información ponderándolas por sus incertidumbres.


In [ ]:
mu_prior = 0.0
P_prior = 4.0
z_measured = 1.2
R = 1.0

P_post = 1.0 / (1.0 / P_prior + 1.0 / R)
mu_post = P_post * (mu_prior / P_prior + z_measured / R)

x_bayes = np.linspace(-6.0, 6.0, 1_500)
prior_pdf = normal_pdf(x_bayes, mu_prior, np.sqrt(P_prior))
likelihood_as_x = normal_pdf(z_measured, x_bayes, np.sqrt(R))
unnormalized = prior_pdf * likelihood_as_x
posterior_numeric = unnormalized / integrate(unnormalized, x_bayes)
posterior_analytic = normal_pdf(x_bayes, mu_post, np.sqrt(P_post))

print(f'Prior:     media={mu_prior:.3f}, varianza={P_prior:.3f}')
print(f'Medición:  z={z_measured:.3f}, R={R:.3f}')
print(f'Posterior: media={mu_post:.3f}, varianza={P_post:.3f}')

assert np.allclose(posterior_numeric, posterior_analytic, atol=2e-5)
assert P_post < P_prior

fig_bayes = go.Figure()
fig_bayes.add_trace(go.Scatter(
    x=x_bayes, y=prior_pdf, mode='lines', line=dict(width=3),
    name='Prior p(x)'
))
fig_bayes.add_trace(go.Scatter(
    x=x_bayes, y=likelihood_as_x, mode='lines', line=dict(width=3, dash='dot'),
    name='Verosimilitud p(z | x)'
))
fig_bayes.add_trace(go.Scatter(
    x=x_bayes, y=posterior_analytic, mode='lines', line=dict(width=4),
    name='Posterior p(x | z)'
))
fig_bayes.update_layout(
    title='Actualización bayesiana gaussiana',
    xaxis_title='Estado x', yaxis_title='Densidad',
    template='plotly_white', width=900, height=500
)
fig_bayes.show()


# 6. Teorema del límite central

Sean $X_1,\ldots,X_n$ variables independientes e idénticamente distribuidas con media finita $\mu$ y varianza finita $\sigma^2$. Para la media muestral

$$\bar X_n=\frac{1}{n}\sum_{i=1}^n X_i,$$

el teorema del límite central afirma que

$$
\frac{\sqrt n(\bar X_n-\mu)}{\sigma}
\xrightarrow{d}\mathcal N(0,1),
$$

o, de manera aproximada para $n$ grande,

$$\bar X_n\approx\mathcal N\!\left(\mu,\frac{\sigma^2}{n}\right).$$

La distribución original **no necesita ser gaussiana**. Usaremos variables exponenciales, claramente asimétricas, con media y desviación estándar iguales a 1. Al promediar más términos, la distribución del promedio se vuelve aproximadamente normal y su desviación disminuye como $1/\sqrt n$.


In [ ]:
sample_sizes = [1, 2, 5, 30]
n_trials = 25_000
mu_exp = 1.0
sigma_exp = 1.0

fig_clt = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'n = {n}' for n in sample_sizes]
)

for index, n in enumerate(sample_sizes):
    row = index // 2 + 1
    col = index % 2 + 1
    means = rng.exponential(scale=1.0, size=(n_trials, n)).mean(axis=1)
    std_clt = sigma_exp / np.sqrt(n)
    grid = np.linspace(0.0, max(4.0, np.percentile(means, 99.8)), 500)

    fig_clt.add_trace(
        go.Histogram(x=means, histnorm='probability density', nbinsx=100,
                     opacity=0.6, showlegend=(index == 0), name='Medias simuladas'),
        row=row, col=col
    )
    fig_clt.add_trace(
        go.Scatter(x=grid, y=normal_pdf(grid, mu_exp, std_clt), mode='lines',
                   line=dict(color='black', width=3), showlegend=(index == 0),
                   name='Aproximación normal'),
        row=row, col=col
    )

    print(
        f'n={n:2d}: media simulada={means.mean():.4f}, '
        f'std simulada={means.std(ddof=1):.4f}, std TLC={std_clt:.4f}'
    )

fig_clt.update_layout(
    title='Teorema del límite central con variables exponenciales',
    template='plotly_white', width=1_000, height=750, barmode='overlay'
)
fig_clt.update_xaxes(title_text='Media muestral')
fig_clt.update_yaxes(title_text='Densidad')
fig_clt.show()


## Qué demuestra y qué no demuestra el límite central

El experimento muestra que la suma o el promedio de muchos efectos independientes con varianza finita puede aproximarse por una gaussiana. Esto ayuda a explicar por qué el ruido agregado de sensores suele modelarse como normal. Sin embargo:

- no prueba que todo ruido físico sea gaussiano;
- no garantiza independencia temporal ni ruido blanco;
- puede converger lentamente para distribuciones muy asimétricas;
- no se aplica en esta forma si la varianza es infinita;
- correlaciones fuertes, sesgos, saturaciones y valores atípicos requieren modelos adicionales.

# Conexión final con el filtro de Kalman

El filtro de Kalman puede leerse como una secuencia de operaciones sobre distribuciones gaussianas:

1. **Predicción:** propaga el valor esperado y la covarianza a través del modelo dinámico.
2. **Condicionamiento:** interpreta el estado a la luz de una nueva medición.
3. **Bayes:** combina la predicción previa con la verosimilitud de la medición.
4. **Resultado:** en el caso lineal-gaussiano, la posterior sigue siendo gaussiana y queda descrita por una media y una covarianza.

## Ejercicios propuestos

1. Calcula numéricamente $P(0\le X\le2)$ para $X\sim\mathcal N(1,0.5^2)$.
2. Cambia el signo de $\Sigma_{12}$ en la gaussiana bivariada y explica la rotación de sus contornos.
3. En el ejemplo del detector, reduce la probabilidad previa del blanco a 1 %. ¿Por qué cambia tanto la posterior?
4. En la actualización gaussiana, prueba $R=0.1$, $1$ y $10$. Relaciona el resultado con la confianza en el sensor.
5. Repite el TLC con distribuciones uniforme y Bernoulli. Verifica la media y la desviación teóricas de $\bar X_n$.
6. Investiga qué sucede con una distribución de Cauchy, cuya media y varianza no están definidas.

## Resumen

Una densidad asigna probabilidad mediante áreas; las gaussianas resumen incertidumbre con media y covarianza; condicionar incorpora información observada; Bayes actualiza creencias; y el límite central explica por qué una normal puede emerger de muchos efectos acumulados. Juntas, estas ideas forman la base probabilística de los filtros de Kalman.
